In [60]:
import sys
import os
import re
import pandas as pd
import importlib
import src.features as F
import pyarrow as pa
import pyarrow.parquet as pq


from dotenv import load_dotenv
from pathlib import Path

%pip install pyarrow
%pip install duckdb fastparquet



Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# Repo root

In [48]:
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Repo root:", REPO)

Repo root: /Users/gokulvs/ICT.projects/AQI_reports/AQI-Air-_pollution_prediction


## loading dataset

In [49]:
import os
import pandas as pd
from pathlib import Path

# 1) Where the data lives
DATA_DIR = Path(os.environ.get("AQI_DATA_DIR", REPO / "data"))
D1 = DATA_DIR / "station_hour_merged.parquet"
print("D1 is:", D1)
assert D1.exists(), f"Cannot find {D1}. Folder contains: {sorted(p.name for p in DATA_DIR.iterdir())}"

KEEP = ["StationId", "City", "Datetime", "AQI",
        "PM2.5", "PM10", "NO2", "NOx", "NH3", "CO", "SO2", "O3",
        "Temperature_C", "Relative_Humidity_pct",
        "Wind_Speed_kmh", "Wind_Direction_deg"]


def load_parquet(path, columns):
    """Try several parquet readers and return the first DataFrame that loads."""
    errors = {}

    # Attempt 1: pandas + pyarrow
    try:
        return pd.read_parquet(path, columns=columns, engine="pyarrow"), "pyarrow"
    except Exception as e:
        errors["pyarrow"] = f"{type(e).__name__}: {str(e)[:150]}"

    # Attempt 2: pandas + fastparquet (a different reader, no pyarrow)
    try:
        return pd.read_parquet(path, columns=columns, engine="fastparquet"), "fastparquet"
    except Exception as e:
        errors["fastparquet"] = f"{type(e).__name__}: {str(e)[:150]}"

    # Attempt 3: DuckDB (independent of pyarrow)
    try:
        import duckdb
        cols = ", ".join(f'"{c}"' for c in columns)
        p = str(path).replace("'", "''")
        return duckdb.sql(f"SELECT {cols} FROM read_parquet('{p}')").df(), "duckdb"
    except Exception as e:
        errors["duckdb"] = f"{type(e).__name__}: {str(e)[:150]}"

    for name, msg in errors.items():
        print(f"  {name} failed -> {msg}")
    raise RuntimeError("No parquet reader worked. See the messages above.")


df, reader = load_parquet(D1, KEEP)
print("loaded with:", reader)

df = df.sort_values(["StationId", "Datetime"]).reset_index(drop=True)

# 2) Sanity checks against what we were told about D1
print("shape:         ", df.shape)                                   # expect (2589083, 16)
print("stations:      ", df["StationId"].nunique())                  # expect 110
print("cities:        ", df["City"].nunique())                       # expect 26
print("date range:    ", df["Datetime"].min(), "->", df["Datetime"].max())
print("AQI missing %: ", round(df["AQI"].isna().mean() * 100, 1))    # expect about 22
print("Datetime type: ", df["Datetime"].dtype)                       # expect datetime64[...]
print("wind dir range:", df["Wind_Direction_deg"].min(), "-", df["Wind_Direction_deg"].max())  # expect 1 - 360

D1 is: /Users/gokulvs/ICT.projects/AQI_reports/AQI-Air-_pollution_prediction/data/station_hour_merged.parquet
loaded with: fastparquet
shape:          (2589083, 16)
stations:       110
cities:         26
date range:     2015-01-01 01:00:00 -> 2020-07-01 00:00:00
AQI missing %:  22.0
Datetime type:  datetime64[ns]
wind dir range: 1 - 360


In [ ]:
importlib.reload(F)
F.assert_contiguous(df)

OK: 1 station(s), 3,320 rows, hourly-contiguous within every station.


In [56]:
TEST_QUANTILE = 0.8
GAP_H = 48
LOCKDOWN_START = pd.Timestamp("2020-03-25")


def get_test_start(df: pd.DataFrame, time_col: str = "Datetime",
                   q: float = TEST_QUANTILE) -> pd.Timestamp:
    """80th percentile of the UNIQUE timestamps (an actual timestamp, no interpolation)."""
    ts = np.sort(df[time_col].unique())
    return pd.Timestamp(ts[int(q * (len(ts) - 1))])


def add_split(df: pd.DataFrame):
    """Add `split` (train/gap/test) and `is_lockdown`. Returns (df, test_start)."""
    out = df.copy()
    test_start = get_test_start(out)
    gap_start = test_start - pd.Timedelta(hours=GAP_H)
    t = out["Datetime"]
    out["split"] = np.select([t < gap_start, t < test_start], ["train", "gap"], default="test")
    out["is_lockdown"] = t >= LOCKDOWN_START
    return out, test_start


def split_report(df: pd.DataFrame, test_start: pd.Timestamp) -> None:
    """Print row counts and date ranges per split, test_start and lockdown share of test."""
    gap_start = test_start - pd.Timedelta(hours=GAP_H)
    print("test_start:", test_start, "| gap_start:", gap_start)
    for name in ["train", "gap", "test"]:
        p = df[df["split"] == name]
        print(f"{name:5s} rows={len(p):>10,}   {p['Datetime'].min()} -> {p['Datetime'].max()}")

    tr, ga, te = (df[df["split"] == s]["Datetime"] for s in ["train", "gap", "test"])
    assert tr.max() < ga.min() and ga.max() < te.min(), "splits overlap in time"

    test = df[df["split"] == "test"]
    n_lock = int(test["is_lockdown"].sum())
    print(f"lockdown share of test rows: {n_lock / len(test) * 100:.1f}%  ({n_lock:,} of {len(test):,})")

In [58]:
import traceback

# 1) Do the things Cell 4 needs actually exist in this kernel?
for name in ["df", "F", "REPO"]:
    print(f"{name} defined:", name in globals())

print("F loaded from:", getattr(F, "__file__", None))
print("F has build_features:", hasattr(F, "build_features"))
print("df shape:", df.shape, "| columns:", list(df.columns))
print("KL007 rows:", int((df["StationId"] == "KL007").sum()))

# 2) Run the test, but print any failure compactly
try:
    df_one = df[df["StationId"] == "KL007"].copy().reset_index(drop=True)
    feat_one = F.build_features(df_one)
    print("\nSUCCESS: rows before", len(df_one), "| rows after", len(feat_one))
    new_cols = [c for c in feat_one.columns if c not in df_one.columns]
    print(len(new_cols), "new columns:", new_cols)
except Exception as e:
    print("\nFAILED:", type(e).__name__, "->", str(e)[:300])
    print("---- last lines of traceback ----")
    print(traceback.format_exc()[-900:])

df defined: True
F defined: True
REPO defined: True
F loaded from: /Users/gokulvs/ICT.projects/AQI_reports/AQI-Air-_pollution_prediction/src/features.py
F has build_features: False
df shape: (2589083, 16) | columns: ['StationId', 'City', 'Datetime', 'AQI', 'PM2.5', 'PM10', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Temperature_C', 'Relative_Humidity_pct', 'Wind_Speed_kmh', 'Wind_Direction_deg']
KL007 rows: 3320

FAILED: AttributeError -> module 'src.features' has no attribute 'build_features'
---- last lines of traceback ----
Traceback (most recent call last):
  File "/var/folders/zg/zss49j013h34jvskzyn6s1jc0000gn/T/ipykernel_22447/3815941488.py", line 15, in <module>
    feat_one = F.build_features(df_one)
               ^^^^^^^^^^^^^^^^
AttributeError: module 'src.features' has no attribute 'build_features'



In [ ]:
# 1) Check the SAVED files before importing anything
feat_path, split_path = REPO / "src" / "features.py", REPO / "src" / "split.py"
for p in (feat_path, split_path):
    assert p.exists(), f"{p} does not exist. Create it (Part A/B) and save it."

feat_funcs = re.findall(r"^def (\w+)", feat_path.read_text(), flags=re.M)
split_funcs = re.findall(r"^def (\w+)", split_path.read_text(), flags=re.M)
print("features.py functions:", feat_funcs)
print("split.py functions:   ", split_funcs)
assert "build_features" in feat_funcs, "Saved features.py lacks build_features. Paste Part A and press Cmd+S."
assert "add_split" in split_funcs, "Saved split.py lacks add_split. Paste Part B and press Cmd+S."

# 2) Forget any cached copy, then import fresh
for name in list(sys.modules):
    if name == "src" or name.startswith("src."):
        del sys.modules[name]
importlib.invalidate_caches()

import src.features as F
import src.split as S
print("loaded:", F.__file__)

# 3) Contiguity check on the full table
F.assert_contiguous(df)

features.py functions: ['assert_contiguous', 'build_features', 'select_columns', 'leakage_test']
split.py functions:    ['get_test_start', 'add_split', 'split_report']
loaded: /Users/gokulvs/ICT.projects/AQI_reports/AQI-Air-_pollution_prediction/src/features.py
OK: 110 station(s), 2,589,083 rows, hourly-contiguous within every station.


# station (KL007)

In [61]:
df_one = df[df["StationId"] == "KL007"].copy().reset_index(drop=True)
feat_one = F.build_features(df_one)

print("rows before:", len(df_one), "| rows after:", len(feat_one))
new_cols = [c for c in feat_one.columns if c not in df_one.columns]
print(len(new_cols), "new columns")                 # expect 19
print(feat_one[new_cols].isna().sum())

F.leakage_test(feat_one, df_one, n=500)             # three PASS lines expected

OK: 1 station(s), 3,320 rows, hourly-contiguous within every station.
rows before: 3320 | rows after: 2863
19 new columns
Target_AQI_24h          159
Target_AQI_48h          267
AQI_lag_1h                9
AQI_lag_6h               48
AQI_lag_12h              89
AQI_lag_24h             159
AQI_lag_48h             267
AQI_rolling_mean_6h      16
AQI_rolling_mean_12h     40
AQI_rolling_mean_24h     77
Hour_sin                  0
Hour_cos                  0
DayOfWeek_sin             0
DayOfWeek_cos             0
Month_sin                 0
Month_cos                 0
DayOfYear                 0
Wind_Dir_sin              0
Wind_Dir_cos              0
dtype: int64
PASS AQI_lag_24h: 500 random rows match a direct lookup
PASS Target_AQI_24h: 500 random rows match a direct lookup
PASS Target_AQI_48h: 500 random rows match a direct lookup


## build features on all 110 stations

In [62]:
feat = F.build_features(df)
print("rows before:", len(df), "| rows after (AQI present):", len(feat))

OK: 110 station(s), 2,589,083 rows, hourly-contiguous within every station.
rows before: 2589083 | rows after (AQI present): 2018893


# Train test split

In [63]:
feat, test_start = S.add_split(feat)
S.split_report(feat, test_start)

test_start: 2019-05-26 12:00:00 | gap_start: 2019-05-24 12:00:00
train rows= 1,144,531   2015-01-01 16:00:00 -> 2019-05-24 11:00:00
gap   rows=     3,601   2019-05-24 12:00:00 -> 2019-05-26 11:00:00
test  rows=   870,761   2019-05-26 12:00:00 -> 2020-07-01 00:00:00
lockdown share of test rows: 25.5%  (222,412 of 870,761)


In [64]:
F.leakage_test(feat, df)                            # 1,000 random rows, three PASS lines

assert feat["AQI"].notna().all(), "rows with missing AQI should have been dropped"
assert not feat.duplicated(["StationId", "Datetime"]).any(), "duplicate keys"
print("split counts:\n", feat["split"].value_counts())

PASS AQI_lag_24h: 1,000 random rows match a direct lookup
PASS Target_AQI_24h: 1,000 random rows match a direct lookup
PASS Target_AQI_48h: 1,000 random rows match a direct lookup
split counts:
 split
train    1144531
test      870761
gap         3601
Name: count, dtype: int64


In [65]:
D2 = F.select_columns(feat)
print("D2 shape:", D2.shape)                        # expect 36 columns

OUT_DIR = DATA_DIR / "processed"
OUT_DIR.mkdir(exist_ok=True)
OUT = OUT_DIR / "features_v1.parquet"


def save_parquet(frame, path):
    """Write parquet with the first writer that works."""
    try:
        frame.to_parquet(path, engine="pyarrow", index=False)
        return "pyarrow"
    except Exception as e:
        print("pyarrow write failed ->", type(e).__name__, str(e)[:120])
    try:
        frame.to_parquet(path, engine="fastparquet", index=False)
        return "fastparquet"
    except Exception as e:
        print("fastparquet write failed ->", type(e).__name__, str(e)[:120])
    import duckdb
    con = duckdb.connect()
    con.register("d2", frame)
    p = str(path).replace("'", "''")
    con.execute(f"COPY d2 TO '{p}' (FORMAT PARQUET)")
    return "duckdb"


writer = save_parquet(D2, OUT)
print("saved with:", writer, "->", OUT)

check, _ = load_parquet(OUT, list(D2.columns))      # load_parquet is defined in Cell 2
print("read back:", check.shape, "| matches:", check.shape == D2.shape)

D2 shape: (2018893, 36)
pyarrow write failed -> ArrowKeyError A type extension with name pandas.period already defined
saved with: fastparquet -> /Users/gokulvs/ICT.projects/AQI_reports/AQI-Air-_pollution_prediction/data/processed/features_v1.parquet
read back: (2018893, 36) | matches: True


In [66]:
import numpy as np

print("columns in Card 2 order:", list(check.columns) == F.FINAL_COLUMNS)
print("Datetime dtype:", check["Datetime"].dtype, "| is_lockdown dtype:", check["is_lockdown"].dtype)

dtype_diff = check.dtypes[check.dtypes != D2.dtypes]
print("dtype differences:", dict(dtype_diff) if len(dtype_diff) else "none")

print("keys identical:       ",
      check[["StationId", "Datetime"]].reset_index(drop=True)
      .equals(D2[["StationId", "Datetime"]].reset_index(drop=True)))
print("split identical:      ", check["split"].reset_index(drop=True).equals(D2["split"].reset_index(drop=True)))
print("is_lockdown identical:", check["is_lockdown"].reset_index(drop=True).equals(D2["is_lockdown"].reset_index(drop=True)))

bad = [c for c in D2.select_dtypes("number").columns
       if not np.allclose(check[c].to_numpy(float), D2[c].to_numpy(float), equal_nan=True)]
print("numeric columns that differ:", bad if bad else "none")

columns in Card 2 order: True
Datetime dtype: datetime64[ns] | is_lockdown dtype: bool
dtype differences: {'StationId': <StringDtype(storage='python', na_value=nan)>, 'City': <StringDtype(storage='python', na_value=nan)>}
keys identical:        False
split identical:       True
is_lockdown identical: True
numeric columns that differ: none


In [67]:
import numpy as np

a, b = D2.reset_index(drop=True), check.reset_index(drop=True)

print("StationId values equal:", (a["StationId"].astype(str).to_numpy() == b["StationId"].astype(str).to_numpy()).all())
print("City values equal:     ", (a["City"].astype(str).to_numpy() == b["City"].astype(str).to_numpy()).all())
print("Datetime values equal: ", (a["Datetime"].to_numpy() == b["Datetime"].to_numpy()).all())

print("D2 dtypes:    ", a["StationId"].dtype, "|", a["City"].dtype)
print("check dtypes: ", b["StationId"].dtype, "|", b["City"].dtype)
print("missing in StationId/City (D2, check):",
      int(a[["StationId", "City"]].isna().sum().sum()), int(b[["StationId", "City"]].isna().sum().sum()))

StationId values equal: True
City values equal:      True
Datetime values equal:  True
D2 dtypes:     object | object
check dtypes:  str | str
missing in StationId/City (D2, check): 0 0
